In [ ]:
import numpy as np
import torch

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
import torch
import torch.nn as nn


class Sine(nn.Module):
    def forward(self, x):
        return torch.sin(x)


class SawSine(nn.Module):
    def forward(self, x):
        t = torch.remainder(x, 1.0)
        return 1 - 4 * torch.abs(t - 0.5)


class SinNoise(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=32, out_dim=1, saw=True):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            Sine() if not saw else SawSine(),
            nn.Linear(hidden_dim, out_dim),
        )
        self._init_with_noise()

    def _init_with_noise(self, std=0.5):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=std)
                nn.init.normal_(module.bias, mean=0.0, std=std)

    def forward(self, x):
        return self.net(x)


model = SinNoise(in_dim=2, hidden_dim=32, out_dim=1, saw=True)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def evaluate_on_grid(model, x_range=(-10, 10), y_range=(-10, 10), resolution=120):
    x = np.linspace(x_range[0], x_range[1], resolution)
    y = np.linspace(y_range[0], y_range[1], resolution)
    xx, yy = np.meshgrid(x, y)
    grid = np.stack([xx.flatten(), yy.flatten()], axis=-1)

    with torch.no_grad():
        inputs = torch.from_numpy(grid).float()
        outputs = model(inputs).numpy().reshape(resolution, resolution)

    return x, y, xx, yy, outputs


def plot_2d_3d(model, x_range=(-10, 10), y_range=(-10, 10), resolution=120):
    x, y, xx, yy, outputs = evaluate_on_grid(
        model, x_range=x_range, y_range=y_range, resolution=resolution
    )

    # 2D heatmap and 3D surface side-by-side
    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "heatmap"}, {"type": "surface"}]],
        subplot_titles=("2D Heatmap", "3D Surface"),
        horizontal_spacing=0.08,
    )

    fig.add_trace(
        go.Heatmap(
            z=outputs, x=x, y=y, colorscale="Viridis", colorbar=dict(title="output")
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Surface(z=outputs, x=xx, y=yy, colorscale="Viridis", showscale=False),
        row=1,
        col=2,
    )

    fig.update_xaxes(title_text="x", row=1, col=1)
    fig.update_yaxes(title_text="y", row=1, col=1)
    fig.update_scenes(
        xaxis_title="x", yaxis_title="y", zaxis_title="output", row=1, col=2
    )
    fig.update_layout(
        height=520, width=1200, title=f"Output over {x_range} x {y_range}"
    )
    fig.show()


s = 10
r = (-s, s)
plot_2d_3d(model, x_range=r, y_range=r, resolution=120)